# LLM-WITHMEM: Level 3 Memory Encoder Training

Train a 7.1M-parameter memory encoder that produces K,V pairs for KV injection into SmolLM2-1.7B-Instruct.

**Architecture**: Profile tokens → Small Transformer Encoder → Perceiver Resampler → Layer-Group K,V Projections → Per-Head Gates → Inject into LLM DynamicCache

**Training**: KL Distillation — match inject-path logits to gold-path (profile-in-system-prompt) logits. Only encoder params update; LLM is frozen.

## 1. GPU Check & Environment Setup

In [ ]:
import torch
import subprocess

assert torch.cuda.is_available(), "No GPU detected — switch to a GPU runtime"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

# Auto-select batch size based on GPU VRAM
if vram_gb >= 70:     # H100 / A100 80GB
    BATCH_SIZE = 8
    GRAD_ACCUM = 1
elif vram_gb >= 38:   # A100 40GB / A6000
    BATCH_SIZE = 4
    GRAD_ACCUM = 2
elif vram_gb >= 14:   # T4 / V100 / RTX 4090
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
else:                 # Smaller GPUs
    BATCH_SIZE = 1
    GRAD_ACCUM = 8

print(f"\nAuto-config: batch_size={BATCH_SIZE}, grad_accum={GRAD_ACCUM}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

## 2. Clone Repository

In [ ]:
import os

REPO_DIR = "/content/LLM-WITHMEM"

if not os.path.exists(REPO_DIR):
    # For private repo: set your GitHub token
    # Option 1: Colab Secrets (recommended)
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GH_TOKEN = ""  # Set manually if not using Colab Secrets

    if GH_TOKEN:
        !git clone https://{GH_TOKEN}@github.com/Pkansagra-hub/LLM-WITHMEM.git {REPO_DIR}
    else:
        # Public or SSH
        !git clone https://github.com/Pkansagra-hub/LLM-WITHMEM.git {REPO_DIR}
else:
    print(f"Repo already exists at {REPO_DIR}")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 3. Install Dependencies

In [ ]:
!pip install -q torch transformers accelerate matplotlib

## 4. Generate Training Data

In [ ]:
from pathlib import Path

data_dir = Path("experiments/level3_encoder/data")

if not (data_dir / "profiles_train.json").exists():
    print("Generating training profiles...")
    !python -m experiments.level3_encoder.data.generate_profiles
else:
    import json
    n_train = len(json.loads((data_dir / "profiles_train.json").read_text()))
    n_val = len(json.loads((data_dir / "profiles_val.json").read_text()))
    print(f"Training data already exists: {n_train} train, {n_val} val profiles")

## 5. Training Configuration

Override defaults here before training. The auto-detected batch size from cell 1 is used.

In [ ]:
# ═══════════════════════════════════════════════════════
#  TRAINING HYPERPARAMETERS — edit these as needed
# ═══════════════════════════════════════════════════════

MAX_STEPS       = 5000     # Total training steps
LEARNING_RATE   = 1e-4     # Peak learning rate (cosine decay)
EVAL_EVERY      = 250      # Validate every N steps
SAVE_EVERY      = 500      # Checkpoint every N steps
WARMUP_STEPS    = 100      # LR warmup steps
MAX_NEW_TOKENS  = 128      # Max tokens for generation
LAMBDA_DISTILL  = 1.0      # KL distillation weight
LAMBDA_KV_ALIGN = 0.1      # KV alignment weight (L2)

print(f"Training config:")
print(f"  max_steps:       {MAX_STEPS}")
print(f"  batch_size:      {BATCH_SIZE}")
print(f"  grad_accum:      {GRAD_ACCUM}")
print(f"  effective_batch:  {BATCH_SIZE * GRAD_ACCUM}")
print(f"  learning_rate:   {LEARNING_RATE}")
print(f"  eval_every:      {EVAL_EVERY}")
print(f"  save_every:      {SAVE_EVERY}")

## 6. Load Model & Initialize Trainer

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from experiments.level3_encoder.config import Config

# Build config with overrides
config = Config()
config.training.batch_size = BATCH_SIZE
config.training.gradient_accumulation_steps = GRAD_ACCUM
config.training.max_steps = MAX_STEPS
config.training.learning_rate = LEARNING_RATE
config.training.eval_every = EVAL_EVERY
config.training.save_every = SAVE_EVERY
config.training.warmup_steps = WARMUP_STEPS
config.training.max_new_tokens = MAX_NEW_TOKENS
config.training.lambda_distill = LAMBDA_DISTILL
config.training.lambda_kv_align = LAMBDA_KV_ALIGN

print(f"Loading LLM: {config.model.model_id}")
dtype = getattr(torch, config.model.dtype)

tokenizer = AutoTokenizer.from_pretrained(config.model.model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    config.model.model_id,
    dtype=dtype,
    device_map=config.device,
)
model.eval()

vram = torch.cuda.memory_allocated() / 1024**3
print(f"LLM VRAM: {vram:.2f} GB")

# Initialize trainer
from experiments.level3_encoder.training.trainer import Trainer

trainer = Trainer(config, model, tokenizer)
print(f"\nEncoder params: {trainer.encoder.param_count()['total']:,}")
print(f"Total VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 7. Train

In [ ]:
history = trainer.train()

print(f"\nTraining complete — {len(history)} steps recorded.")
print(f"Checkpoints: {config.output_dir}")
print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

## 8. Training Curves

In [ ]:
import matplotlib.pyplot as plt
import json

# Load history from saved log if needed
if not history:
    log_path = Path(config.log_dir) / "training_history.json"
    if log_path.exists():
        history = json.loads(log_path.read_text())

steps = [h["step"] for h in history]
train_loss = [h["loss"] for h in history]

# Validation points
val_steps = [h["step"] for h in history if "val_loss" in h]
val_loss = [h["val_loss"] for h in history if "val_loss" in h]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(steps, train_loss, alpha=0.4, label="Train loss (per step)")

# Smoothed train loss
window = min(50, len(train_loss) // 5) if len(train_loss) > 10 else 1
if window > 1:
    import numpy as np
    smoothed = np.convolve(train_loss, np.ones(window)/window, mode='valid')
    ax.plot(steps[window-1:], smoothed, label=f"Train loss (smoothed, w={window})", linewidth=2)

if val_loss:
    ax.plot(val_steps, val_loss, 'ro-', label="Val loss", markersize=6, linewidth=2)

ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Memory Encoder Training")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if val_loss:
    print(f"Best val loss: {min(val_loss):.4f} at step {val_steps[val_loss.index(min(val_loss))]}")

## 9. Evaluate Best Checkpoint

In [ ]:
from experiments.level3_encoder.model.encoder import MemoryEncoder
from experiments.level3_encoder.evaluation.evaluator import (
    evaluate_encoder,
    print_eval_summary,
)
from experiments.level3_encoder.data.dataset import ProfileQueryDataset

# Load best checkpoint
ckpt_path = Path(config.output_dir) / "encoder_best.pt"
if not ckpt_path.exists():
    ckpt_path = Path(config.output_dir) / "encoder_final.pt"

print(f"Loading checkpoint: {ckpt_path}")
checkpoint = torch.load(ckpt_path, map_location=config.device, weights_only=True)

encoder = MemoryEncoder(
    embedding_layer=model.get_input_embeddings(),
    llm_embed_dim=config.model.hidden_size,
    d_model=config.encoder.d_model,
    n_heads=config.encoder.n_heads,
    n_layers=config.encoder.n_layers,
    num_memory_slots=config.encoder.num_memory_slots,
    num_llm_layers=config.model.num_layers,
    num_kv_heads=config.model.num_kv_heads,
    head_dim=config.model.head_dim,
    num_layer_groups=config.encoder.num_layer_groups,
    gate_init_bias=config.encoder.gate_init_bias,
    dropout=config.encoder.dropout,
).to(config.device)

encoder.load_state_dict(checkpoint["encoder_state_dict"])
encoder.eval()

print(f"Loaded from step {checkpoint.get('step', '?')}, "
      f"val_loss={checkpoint.get('val_loss', '?')}")

# Evaluate
val_dataset = ProfileQueryDataset(
    profiles_path=str(data_dir / "profiles_val.json"),
    queries_path=str(data_dir / "queries.json"),
    tokenizer=tokenizer,
    max_profile_tokens=config.encoder.max_profile_tokens,
    seed=config.training.seed + 1,
)

results = evaluate_encoder(
    encoder=encoder,
    llm=model,
    tokenizer=tokenizer,
    dataset=val_dataset,
    config=config,
    max_samples=50,
)

print_eval_summary(results)

## 10. Sample Outputs — Gold vs Inject

In [ ]:
from experiments.level3_encoder.model.injector import (
    generate_with_injection,
)

NUM_SAMPLES = 5

encoder.eval()
for i in range(min(NUM_SAMPLES, len(val_dataset))):
    sample = val_dataset[i]
    profile_text = sample["profile_text"]
    gold_prompt = sample["gold_prompt"]
    suffix = sample["inject_suffix"]
    keywords = sample["keywords"]

    # Gold generation
    gold_ids = tokenizer(gold_prompt, return_tensors="pt")["input_ids"].to(config.device)
    with torch.no_grad():
        gold_out = model.generate(
            gold_ids, max_new_tokens=config.training.max_new_tokens, do_sample=False
        )
    gold_text = tokenizer.decode(gold_out[0][gold_ids.shape[1]:], skip_special_tokens=True)

    # Inject generation
    profile_ids = sample["profile_input_ids"].unsqueeze(0).to(config.device)
    profile_mask = (profile_ids != tokenizer.pad_token_id).long()
    with torch.no_grad():
        kv_pairs = encoder(profile_ids, profile_mask)
    suffix_ids = tokenizer(suffix, return_tensors="pt")["input_ids"].to(config.device)
    inject_text = generate_with_injection(
        model, tokenizer, suffix_ids, kv_pairs,
        num_memory_slots=config.encoder.num_memory_slots,
        max_new_tokens=config.training.max_new_tokens,
    )

    print(f"{'='*72}")
    print(f"SAMPLE {i+1}")
    print(f"{'='*72}")
    print(f"Profile: {profile_text[:200]}...")
    print(f"Keywords: {keywords}")
    print(f"\n--- GOLD ---")
    print(gold_text[:300])
    print(f"\n--- INJECT ---")
    print(inject_text[:300])
    print()

## 11. Download Checkpoint

In [ ]:
import shutil

output_dir = Path(config.output_dir)
checkpoints = sorted(output_dir.glob("encoder_*.pt"))
print("Available checkpoints:")
for cp in checkpoints:
    size_mb = cp.stat().st_size / 1024**2
    print(f"  {cp.name} ({size_mb:.1f} MB)")

# Pack all outputs for download
archive_name = "encoder_checkpoints"
shutil.make_archive(archive_name, "zip", output_dir)
print(f"\nPacked: {archive_name}.zip")

# Download in Colab
try:
    from google.colab import files
    files.download(f"{archive_name}.zip")
except ImportError:
    print(f"Not in Colab — checkpoint zip at: {os.path.abspath(archive_name)}.zip")